# Manual tests for Max Fan speeds

PWM is a computer signal which changes immediately, while the actual speed of the fan is limited by inertia and takes a few seconds to reach the intended speed. As such, any time the fans change speed, the estimated max_rpm will be nonsensical.

Essentially, this is a sync issue between PWM signals and RPM readings. To avoid this, the code ignores 5 consecutive errors and replaces the values with the correct value. The aim of this script is to check if that is enough to ignore the sync issue.

In [ ]:
import matplotlib.pyplot as plt
import tango
import time
import json

ral_space = 1
subrack = tango.DeviceProxy(f"low-mccs/subrack/stfc-ral-{ral_space}-sr{ral_space}")
subrack.AdminMode = 0
subrack.state()

In [ ]:

# Py plot holders
time_values = []
pwm_values = {
    'fan 1': [],
    'fan 2': [],
    'fan 3': [],
    'fan 4': [],
}
rpm_values =  {
    'fan 1': [],
    'fan 2': [],
    'fan 3': [],
    'fan 4': [],
}
max_rpm_values =  {
    'fan 1': [],
    'fan 2': [],
    'fan 3': [],
    'fan 4': [],
}

def split_values(out_dict: dict, in_list: list):
    """
    Split a list of 4 values to the correct fan id
    """
    for i, val in enumerate(in_list):
        out_dict[f"fan {i+1}"].append(val)

init_time = time.time()
poll_time = 5 # Tis values should match the poll rate of the subrack, seconds
switch_length = 30 # This is how often we change the fan speed, seconds

for pwm in [0, 100, 0, 100]:
    code, result = subrack.SetSubrackFanSpeed(json.dumps({"subrack_fan_id": 1, "speed_percent": pwm}))
    print(f"{code} {result}")

    for i in range(0, round(switch_length/poll_time)):
        # could probably do with a callback, but this is good enough
        print(f"{subrack.state()} {subrack.healthState}")
        current_time = time.time()-init_time

        split_values(pwm_values, subrack.subrackFanSpeedsPercent)
        split_values(rpm_values, subrack.subrackFanSpeeds)
        split_values(max_rpm_values, subrack.subrackMaxFanSpeeds)
        time_values.append(current_time)

        time.sleep(poll_time)

# Plot the result, makes it a lot easier to spot errors.
fig, ax1 = plt.subplots()

ax1.set_xlabel('time (s)')
ax1.set_ylabel('PWM', color="tab:red")
ax1.plot(time_values, pwm_values["fan 1"], color="tab:red")
ax1.tick_params(axis='y', labelcolor="tab:red")

ax2 = ax1.twinx()

ax2.set_ylabel('rpm', color='tab:blue') 
ax2.plot(time_values, rpm_values["fan 1"], color='tab:blue')
ax2.plot(time_values, max_rpm_values["fan 1"], color='tab:purple')
ax2.tick_params(axis='y', labelcolor='tab:blue')
ax2.set_ylim(0, 10000)

fig.tight_layout()
plt.show()